# Crypto Sentiment NLP Model — Naive Bayes

**Deliverable for the Advanced Fintech Synthesis assignment, Part A (Alternative Data → AI).**

This notebook trains the sentiment classifier that powers the *Alt-Data Lab* page in the CoinWise AI MVP. The final cell writes the trained weights into `../api/_lib/ai/nlp/model.ts` — the live TypeScript runtime imports it without any extra wiring.

## Pipeline

1. Load hand-labeled crypto sentiment dataset (211 docs)
2. Exploratory Data Analysis — class distribution, length stats, top words
3. Pre-processing — tokenize, lowercase, stopword removal
4. Stratified train / test split (80/20, seed = 42)
5. Vectorize with bag-of-words (`CountVectorizer`)
6. Train **Multinomial Naive Bayes** with Laplace smoothing (α = 1.0)
7. Evaluate — accuracy, precision, recall, F1, confusion matrix, errors
8. Compare with Logistic Regression baseline
9. Qualitative test on unseen text
10. Export weights → TS runtime

## Why Multinomial Naive Bayes

It is the canonical text classifier in the *Introduction to Information Retrieval* curriculum (Manning, Raghavan & Schütze, 2008, Ch. 13) and the standard baseline against which more elaborate models are judged in fintech NLP courses. It is also interpretable: each prediction is decomposable into per-token contributions, which we surface in the MVP UI.

## 0. Setup

In [ ]:
import json
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Environment ready.')

## 1. Load the dataset (gold + optional silver)

We follow a **distant-supervision** workflow:

- `crypto_sentiment_dataset.json` — **gold** (211 hand-labeled docs)
- `silver_labeled_corpus.json` — **silver** (~500 VADER-auto-labeled docs scraped from Hacker News and crypto RSS feeds, balanced and crypto-relevance filtered)

Generate silver from the project root with `npm run scrape:corpus && npm run label:corpus`. If the silver file is missing, training falls back to gold-only.

**Honest evaluation rule**: the test set is **gold-only** (hand-verified labels). Silver only joins the training set — it would be unfair to evaluate against VADER's own auto-labels.

In [ ]:
GOLD_PATH = Path('../data/crypto_sentiment_dataset.json')
SILVER_PATH = Path('../data/silver_labeled_corpus.json')

with GOLD_PATH.open(encoding='utf-8') as f:
    gold = pd.DataFrame(json.load(f))
print(f'Gold dataset: {len(gold)} hand-labeled docs from {GOLD_PATH.name}')

if SILVER_PATH.exists():
    with SILVER_PATH.open(encoding='utf-8') as f:
        silver_raw = json.load(f)
    silver = pd.DataFrame([{'text': d['text'], 'label': d['label']} for d in silver_raw])
    print(f'Silver corpus: {len(silver)} VADER-auto-labeled docs from {SILVER_PATH.name}')
    print(f'  silver class distribution: {dict(silver["label"].value_counts())}')
else:
    silver = pd.DataFrame(columns=['text', 'label'])
    print(f'Silver corpus: NOT FOUND — falling back to gold-only training.')

# For EDA we look at gold only — silver is for training augmentation, not analysis.
df = gold.copy()
df.head(5)

## 2. Exploratory Data Analysis

### 2.1 Class distribution

A roughly balanced corpus avoids the classifier biasing toward the majority class. We aim for ~80 / 80 / 50 (positive / negative / neutral).

In [ ]:
class_counts = df['label'].value_counts()
print(class_counts)
print(f"\nBalance ratio (max / min) = {class_counts.max() / class_counts.min():.2f}")

palette = {'positive': '#10b981', 'negative': '#ef4444', 'neutral': '#94a3b8'}
fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind='bar', color=[palette[c] for c in class_counts.index], ax=ax)
ax.set_title(f'Class distribution (n={len(df)})')
ax.set_ylabel('Number of documents')
ax.set_xlabel('Label')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.2 Document length

Headlines (short) vs. forum posts (long) have very different sentiment density. Useful to see before choosing the vectorization strategy.

In [ ]:
df['n_words'] = df['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
for label, color in palette.items():
    df.loc[df['label'] == label, 'n_words'].plot(
        kind='hist', bins=20, alpha=0.55, color=color, label=label, ax=ax,
    )
ax.set_title('Word count distribution per class')
ax.set_xlabel('Words per document')
ax.set_ylabel('Documents')
ax.legend()
plt.tight_layout()
plt.show()

df.groupby('label')['n_words'].describe()[['mean', 'std', 'min', 'max']]

### 2.3 Most common words per class

A sanity check that labels match the text. Positive should surface *rally, surge, approval*; negative should surface *hack, crash, rug*; neutral should be mostly neutral nouns.

In [ ]:
def quick_tokens(text):
    return [w for w in re.findall(r"[a-z']+", text.lower()) if len(w) >= 3]

word_counts = {c: Counter() for c in df['label'].unique()}
for _, row in df.iterrows():
    word_counts[row['label']].update(quick_tokens(row['text']))

for c in ['positive', 'negative', 'neutral']:
    print(f"\n=== Top 12 in '{c}' ===")
    for w, n in word_counts[c].most_common(12):
        print(f"  {w:20s} {n}")

## 3. Preprocessing pipeline

This pipeline matches the TypeScript runtime byte-for-byte so the trained model classifies live HN / Reddit text the same way it did at training time:

1. lowercase
2. strip URLs
3. strip markdown punctuation
4. tokenize on alpha + apostrophe boundaries  (keeps *don't*, *it's*)
5. length filter `2 ≤ len ≤ 20`
6. drop English stopwords (40 of the most common function words)

In [ ]:
STOPWORDS = {
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'shall',
    'should', 'can', 'could', 'may', 'might', 'must', 'and', 'or', 'but',
    'if', 'then', 'else', 'when', 'while', 'as', 'of', 'in', 'on', 'at',
    'to', 'for', 'from', 'by', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'it',
    'we', 'they', 'them', 'their', 'my', 'your', 'his', 'her', 'its',
    'our', 'us', 'me', 'him', 'who', 'what', 'which', 'whom', 'whose',
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'[*_`>#~]', ' ', text)
    tokens = re.findall(r"[a-z']+", text)
    return [t for t in tokens if 2 <= len(t) <= 20 and t not in STOPWORDS]

sample = df['text'].iloc[0]
print(f'Original  : {sample}')
print(f'Tokenized : {preprocess(sample)}')

## 4. Stratified train / test split

20 % of each class is held out for evaluation. `random_state=42` makes the split reproducible — re-running the notebook produces identical metrics.

In [ ]:
# Stratified split on GOLD only so the test set is fully hand-verified.
X_train_gold, X_test, y_train_gold, y_test = train_test_split(
    gold['text'].values,
    gold['label'].values,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=gold['label'].values,
)

# Silver joins the TRAINING set only.
if len(silver) > 0:
    X_train = np.concatenate([X_train_gold, silver['text'].values])
    y_train = np.concatenate([y_train_gold, silver['label'].values])
    print(f'Training set : {len(X_train)} docs  ({len(X_train_gold)} gold + {len(silver)} silver)')
else:
    X_train, y_train = X_train_gold, y_train_gold
    print(f'Training set : {len(X_train)} docs  (gold-only — silver corpus not loaded)')

print(f'Test set     : {len(X_test)} docs  (gold-only, hand-verified)')
print()
print(f'Train class counts: {dict(Counter(y_train))}')
print(f'Test  class counts: {dict(Counter(y_test))}')

## 5. Vectorize (bag of words)

`CountVectorizer` builds a sparse document-term matrix where each cell is the raw count `f(t, d)`. Multinomial Naive Bayes consumes these raw counts directly — no TF-IDF reweighting needed.

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=preprocess,
    token_pattern=None,
    lowercase=False,
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

vocab = vectorizer.get_feature_names_out()
rows, cols = X_train_vec.shape
sparsity = 100 * (1 - X_train_vec.nnz / (rows * cols))
print(f'Vocabulary size : {len(vocab)}')
print(f'Train matrix    : {X_train_vec.shape}  (rows = docs, cols = tokens)')
print(f'Sparsity        : {sparsity:.2f}% zeros')
print(f'\nFirst 15 vocab terms: {vocab[:15].tolist()}')

## 6. Train Multinomial Naive Bayes

Given a document `d` and class `c`,

$$P(c \mid d) \;\propto\; P(c) \cdot \prod_{t \in d} P(t \mid c)^{f(t, d)}$$

With add-α Laplace smoothing,

$$P(t \mid c) = \frac{\text{count}(t, c) + \alpha}{\sum_{t'} \text{count}(t', c) + \alpha \cdot |V|}$$

We work in log space for numerical stability:

$$\log P(c \mid d) = \log P(c) + \sum_{t \in d} f(t, d) \cdot \log P(t \mid c)$$

In [ ]:
clf = MultinomialNB(alpha=1.0)
clf.fit(X_train_vec, y_train)

print(f'Trained MultinomialNB(alpha={clf.alpha})')
print(f'Classes : {clf.classes_.tolist()}')
print()
print('Learned log priors:')
for c, lp in zip(clf.classes_, clf.class_log_prior_):
    print(f'  log P({c:9s}) = {lp:+.4f}   (P = {np.exp(lp):.4f})')

## 7. Evaluate on the held-out test set

### 7.1 Aggregate metrics

In [ ]:
y_pred = clf.predict(X_test_vec)
y_proba = clf.predict_proba(X_test_vec)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy : {acc * 100:.2f}%')
print(f'Macro F1 : {macro_f1 * 100:.2f}%')
print()
print(classification_report(y_test, y_pred, digits=4))

### 7.2 Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=clf.classes_, yticklabels=clf.classes_,
    cbar=False, square=True, ax=ax,
)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title(f'Confusion matrix — Multinomial NB  (acc {acc*100:.1f}% / macro F1 {macro_f1*100:.1f}%)')
plt.tight_layout()
plt.show()

### 7.3 Most decisive tokens per class

For each class `c`, we rank tokens by `log P(t | c)`. The top-ranked tokens are what the classifier *looks for* when it predicts class `c`.

In [ ]:
TOP_K = 15
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, c in enumerate(clf.classes_):
    log_p = clf.feature_log_prob_[i]
    top_idx = np.argsort(log_p)[-TOP_K:][::-1]
    top_tokens = [vocab[j] for j in top_idx]
    top_vals = log_p[top_idx]
    color = palette.get(c, '#888')
    axes[i].barh(range(TOP_K)[::-1], top_vals, color=color, alpha=0.85)
    axes[i].set_yticks(range(TOP_K)[::-1])
    axes[i].set_yticklabels(top_tokens, fontfamily='monospace')
    axes[i].set_xlabel('log P(t | c)')
    axes[i].set_title(f"Top {TOP_K} tokens — '{c}'")
plt.tight_layout()
plt.show()

### 7.4 Error analysis

Which test documents did the classifier get wrong? These reveal the model's blind spots — useful guidance for growing the training corpus.

In [ ]:
errors = []
for text, true_label, pred_label, proba in zip(X_test, y_test, y_pred, y_proba):
    if true_label != pred_label:
        errors.append({
            'text': text,
            'trueLabel': str(true_label),
            'predicted': str(pred_label),
            'confidence': float(proba.max()),
        })

print(f'Misclassified: {len(errors)} / {len(y_test)}\n')
err_df = pd.DataFrame(errors)
if len(err_df):
    err_df['text'] = err_df['text'].str.slice(0, 90) + '…'
err_df

## 8. Baseline comparison — Logistic Regression

Naive Bayes is the canonical generative text classifier. Logistic Regression on the same bag-of-words features is the standard discriminative baseline. On small, balanced datasets they are usually within a few points of each other.

In [ ]:
lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
lr.fit(X_train_vec, y_train)
lr_pred = lr.predict(X_test_vec)
lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred, average='macro')
print(f'Multinomial Naive Bayes : acc {acc*100:5.2f}%   macro F1 {macro_f1*100:5.2f}%')
print(f'Logistic Regression     : acc {lr_acc*100:5.2f}%   macro F1 {lr_f1*100:5.2f}%')

## 9. Inference demo on unseen text

These three strings are not present in the dataset. We check that the classifier behaves sensibly.

In [ ]:
def classify(text, model=clf):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    print(f'Text : {text}')
    print(f'  → predicted: {pred}')
    for c, p in zip(model.classes_, proba):
        print(f'    P({c:9s}) = {p*100:5.1f}%')
    print()

classify('Bitcoin spot ETF receives official SEC approval, market rallies')
classify('Massive rug pull, founders dumped 80 percent of supply overnight')
classify('Bitcoin closes the day flat at 65000, change less than 1 percent')

## 10. Export trained model → TypeScript runtime

The TS runtime expects an `NbModel` shape: per-class log priors, per-token log likelihoods, per-class OOV fallback, plus the vocabulary. We pull those from sklearn's `feature_log_prob_` / `class_log_prior_` / `feature_count_` and recompute the OOV fallback from the smoothing α.

**Two files are written**:

- `../api/_lib/ai/nlp/model.ts` — weights consumed by `classifier.ts`
- `../api/_lib/ai/nlp/model-metrics.ts` — metrics consumed by the *Alt-Data Lab* UI card

After running this cell, from the project root run `npm run build:api` to rebundle the Vercel function with the new weights.

In [ ]:
classes_list = clf.classes_.tolist()
vocab_list = vocab.tolist()
vocab_size = len(vocab_list)
alpha = float(clf.alpha)

class_doc_count = {c: int(clf.class_count_[i]) for i, c in enumerate(classes_list)}
class_token_count = {c: int(clf.feature_count_[i].sum()) for i, c in enumerate(classes_list)}
log_prior = {c: float(clf.class_log_prior_[i]) for i, c in enumerate(classes_list)}

log_likelihood = {}
for j, t in enumerate(vocab_list):
    row = {}
    for i, c in enumerate(classes_list):
        row[c] = round(float(clf.feature_log_prob_[i, j]), 6)
    log_likelihood[t] = row

oov_log_likelihood = {
    c: round(float(np.log(alpha / (class_token_count[c] + alpha * vocab_size))), 6)
    for c in classes_list
}

trained_at = datetime.utcnow().isoformat() + 'Z'
model_dict = {
    'version': '1.0.0-sklearn',
    'algorithm': 'multinomial-naive-bayes',
    'smoothingAlpha': alpha,
    'classes': classes_list,
    'classDocCount': class_doc_count,
    'classTokenCount': class_token_count,
    'logPrior': log_prior,
    'logLikelihood': log_likelihood,
    'oovLogLikelihood': oov_log_likelihood,
    'vocabulary': vocab_list,
    'trainedAt': trained_at,
    'trainSize': int(len(X_train)),
    'testSize': int(len(X_test)),
}

OUT_MODEL = Path('../api/_lib/ai/nlp/model.ts')
ts_header = (
    '// AUTO-GENERATED by notebooks/train_sentiment_model.ipynb — DO NOT EDIT by hand.\n'
    '// Re-run the notebook to retrain. Source: scikit-learn MultinomialNB.\n'
    "import type { NbModel } from './training/trainer';\n"
)
OUT_MODEL.write_text(
    ts_header + 'export const MODEL: NbModel = ' + json.dumps(model_dict) + ';\n',
    encoding='utf-8',
)
print(f'✓ wrote {OUT_MODEL}  ({OUT_MODEL.stat().st_size} bytes)')

### Export evaluation metrics

`model-metrics.ts` powers the *Training summary* card in the MVP's Alt-Data Lab page.

In [ ]:
report_dict = classification_report(y_test, y_pred, output_dict=True, digits=4)
per_class = {
    c: {
        'precision': round(float(report_dict[c]['precision']), 4),
        'recall':    round(float(report_dict[c]['recall']), 4),
        'f1':        round(float(report_dict[c]['f1-score']), 4),
        'support':   int(report_dict[c]['support']),
    }
    for c in classes_list
}
confusion_dict = {
    c: {c2: int(cm[i, j]) for j, c2 in enumerate(classes_list)}
    for i, c in enumerate(classes_list)
}
metrics_payload = {
    'accuracy': round(float(acc), 4),
    'macroF1': round(float(macro_f1), 4),
    'perClass': per_class,
    'confusion': confusion_dict,
    'testSize': int(len(y_test)),
    'errors': errors[:50],
    'vocabSize': vocab_size,
    'trainSize': int(len(X_train)),
    'trainedAt': trained_at,
    'algorithm': 'multinomial-naive-bayes (scikit-learn)',
    'smoothingAlpha': alpha,
}
OUT_METRICS = Path('../api/_lib/ai/nlp/model-metrics.ts')
metrics_ts_header = (
    '// AUTO-GENERATED by notebooks/train_sentiment_model.ipynb — DO NOT EDIT by hand.\n'
    "import type { EvalMetrics } from './training/trainer';\n"
    'export const MODEL_METRICS: EvalMetrics & {\n'
    '  vocabSize: number; trainSize: number; testSize: number;\n'
    '  trainedAt: string; algorithm: string; smoothingAlpha: number;\n'
    '} = '
)
OUT_METRICS.write_text(
    metrics_ts_header + json.dumps(metrics_payload) + ';\n',
    encoding='utf-8',
)
print(f'✓ wrote {OUT_METRICS}  ({OUT_METRICS.stat().st_size} bytes)')

## Done

From the project root, run `npm run build:api` to rebundle the Vercel function, then open the MVP at **AI → Alt-Data Lab**. The *Training summary* card now reads the metrics you just produced, and the interactive classifier uses these weights.

**Verify** by hitting `http://localhost:3001/api/v1/ai/alt-data/model/info` — the `algorithm` field will read `multinomial-naive-bayes (scikit-learn)`, confirming the runtime is serving the Python-trained model.